# Notebook 03 — Chunking Experiments for DocuTrace

**Goal:** understand chunking by implementing several strategies, inspecting their output, and comparing their trade-offs before selecting anything for the final application.

**Prerequisite:** `data/processed/cleaned_pages.json` created by Notebook 02.

## Learning outcomes

By the end, you should be able to explain:
- why RAG uses chunks instead of embedding one entire PDF;
- chunk size, overlap, boundaries, and metadata;
- character-, word-, sentence-, paragraph-, recursive-, and token-based chunking;
- why one strategy does not win for every document;
- why source/page metadata must travel with every chunk.

We will keep each page separate for now. This makes source citations easy to reason about. Later, we can test chunks that cross page boundaries and structure-aware/semantic chunking.

> The small sizes used below are for visible experiments on our short synthetic PDFs. They are **not** proposed production defaults.

## 1. What is a chunk?

A **chunk** is a smaller section of text that we can later embed and retrieve. Imagine a 100-page PDF that explains scheduling, Hive queries, incident recovery, and SQL validation. Embedding the whole PDF as one unit gives the retriever only one large representation. Smaller chunks let it retrieve a relevant procedure without pulling in the entire document.

Our eventual RAG flow is:

`PDF → extracted pages → cleaned pages → chunks → embeddings → index → retrieve relevant chunks → answer with evidence`

At this stage, we are learning the **chunks** step only. We are not generating embeddings or calling an LLM yet.

### Three parameters to understand

- **Chunk size:** the target maximum length of one chunk. Length may mean characters, words, or model tokens, depending on the strategy.
- **Chunk overlap:** text deliberately repeated between adjacent chunks. It can help preserve context around a boundary, but increases storage/index size and can create redundant retrieved passages.
- **Boundary:** where one chunk stops and another starts. A boundary through the middle of a sentence or procedure may weaken context.

There is a trade-off:
- Too-small chunks may lose necessary context and create many tiny records.
- Too-large chunks may mix unrelated topics and return excess text.
- Too much overlap increases duplication; too little can separate a question from its needed context.

We will measure rather than guess.

## 2. Load the cleaned page records

Each record should contain `source`, `title`, `page_number`, and `text`. We will read from JSON so this notebook does not depend on variables from a previous notebook's memory.

In [1]:
from pathlib import Path
import json
import re
import statistics

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

cleaned_path = PROJECT_ROOT / "data" / "processed" / "cleaned_pages.json"

with cleaned_path.open("r", encoding="utf-8") as file:
    cleaned_pages = json.load(file)

print("Project root:", PROJECT_ROOT)
print("Loaded page records:", len(cleaned_pages))
print("First record keys:", list(cleaned_pages[0].keys()))
print("First source/page:", cleaned_pages[0]["source"], cleaned_pages[0]["page_number"])

Project root: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag
Loaded page records: 4
First record keys: ['source', 'title', 'page_number', 'text']
First source/page: data_pipeline_runbook.pdf 1


If the file cannot be found, check that this notebook is saved inside the project's `notebooks` folder and that Notebook 02 successfully wrote `cleaned_pages.json` to `data/processed/`.

## 3. One standard output shape for every strategy

Chunking functions return text strings. Before using them in RAG, we wrap each string in a record that preserves its source, page number, chunk index, strategy, and length. This metadata will be important for citations and debugging.

In [2]:
def build_chunk_records(pages, splitter, strategy_name):
    """Split page records and preserve source/page metadata on every chunk.

    splitter: a function that accepts one text string and returns list[str].
    """
    records = []

    for page_record in pages:
        page_text = page_record["text"].strip()
        if not page_text:
            continue

        pieces = splitter(page_text)
        pieces = [piece.strip() for piece in pieces if piece and piece.strip()]

        for chunk_index, chunk_text in enumerate(pieces, start=1):
            source = page_record["source"]
            page_number = page_record["page_number"]

            records.append({
                "chunk_id": f"{Path(source).stem}::p{page_number}::c{chunk_index:03d}",
                "source": source,
                "title": page_record.get("title", Path(source).stem),
                "page_number": page_number,
                "chunk_index": chunk_index,
                "strategy": strategy_name,
                "text": chunk_text,
                "char_count": len(chunk_text),
                "word_count": len(chunk_text.split()),
            })

    return records


def show_chunks(chunks, limit=3):
    """Print a compact, readable preview of chunk records."""
    for chunk in chunks[:limit]:
        print(f"\n{chunk['chunk_id']} | {chunk['char_count']} chars | {chunk['word_count']} words")
        print(chunk["text"])

## 4. Strategy A — Fixed-character chunks

This is the simplest baseline: take a window of `chunk_size` characters, move forward by `chunk_size - overlap`, and repeat.

Example with `chunk_size=100` and `overlap=20`:
- chunk 1 covers characters 0–99;
- chunk 2 starts at character 80, so characters 80–99 appear in both chunks.

**Advantages:** simple, predictable, easy to implement.

**Limitations:** may cut a sentence, list item, or word at an awkward place. Character count is not the same as token count.

In [3]:
def fixed_character_chunks(text: str, chunk_size: int = 180, overlap: int = 30) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(text), step):
        piece = text[start:start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        if start + chunk_size >= len(text):
            break

    return chunks

char_chunks = build_chunk_records(
    cleaned_pages,
    lambda text: fixed_character_chunks(text, chunk_size=180, overlap=30),
    "fixed_character",
)

print("Fixed-character chunks:", len(char_chunks))
show_chunks(char_chunks, limit=3)

Fixed-character chunks: 12

data_pipeline_runbook::p1::c001 | 180 chars | 21 words
DATA PIPELINE RUNBOOK
Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.
Normal operation:
1. Confirm the upstream ingestion comp

data_pipeline_runbook::p1::c002 | 180 chars | 25 words
rm the upstream ingestion completed successfully.
2. Confirm the expected business-date partition exists.
3. Start the downstream load after dependencies are healthy.
Owner: Data O

data_pipeline_runbook::p1::c003 | 40 chars | 6 words
ies are healthy.
Owner: Data Operations.


**Inspect it:** notice whether a chunk ends halfway through a sentence or procedure. That is not a coding error; it is a limitation of this boundary strategy.

## 5. Strategy B — Fixed-word chunks

Here, we split on whitespace and group a fixed number of words. Punctuation remains attached to each word, but the method normalizes spacing when it joins the words back together.

**Advantages:** more language-like than counting raw characters in some cases.

**Limitations:** words are not equal in model-token cost; line breaks and original spacing are lost; a word-count limit does not reliably enforce an LLM token budget.

In [4]:
def fixed_word_chunks(text: str, chunk_size: int = 35, overlap: int = 5) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size")

    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        piece = " ".join(words[start:start + chunk_size]).strip()
        if piece:
            chunks.append(piece)
        if start + chunk_size >= len(words):
            break

    return chunks

word_chunks = build_chunk_records(
    cleaned_pages,
    lambda text: fixed_word_chunks(text, chunk_size=35, overlap=5),
    "fixed_word",
)

print("Fixed-word chunks:", len(word_chunks))
show_chunks(word_chunks, limit=3)

Fixed-word chunks: 8

data_pipeline_runbook::p1::c001 | 290 chars | 35 words
DATA PIPELINE RUNBOOK Pipeline: CUSTOMER_DAILY_LOAD Schedule: Daily at 02:00 UTC. Upstream dependency: CUSTOMER_RAW_INGEST. Normal operation: 1. Confirm the upstream ingestion completed successfully. 2. Confirm the expected business-date partition exists. 3. Start the downstream load after

data_pipeline_runbook::p1::c002 | 81 chars | 11 words
Start the downstream load after dependencies are healthy. Owner: Data Operations.

data_pipeline_runbook::p2::c001 | 247 chars | 35 words
DATA PIPELINE RUNBOOK - INCIDENT RESPONSE Symptom: CUSTOMER_DAILY_LOAD failed because an upstream partition is missing. Investigation: 1. Check the upstream job status and logs. 2. Confirm the expected business date and partition format. 3. Verify


## 6. Strategy C — Sentence-aware chunks

This approach first identifies sentence-like units, then combines nearby units until the character budget would be exceeded. It tries to keep a sentence intact.

Our implementation uses a lightweight regular expression. It is an educational heuristic, not a perfect linguistic sentence tokenizer: abbreviations, decimals, numbered lists, code, and some languages can require more robust parsing. Always inspect extracted PDF text before trusting a sentence splitter.

In [5]:
def sentence_units(text: str) -> list[str]:
    # Split after sentence punctuation followed by whitespace, or at line breaks.
    # This is intentionally simple; it will not handle every abbreviation or document layout.
    units = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [unit.strip() for unit in units if unit and unit.strip()]


def pack_units(units: list[str], max_chars: int = 180, separator: str = " ") -> list[str]:
    """Pack units together without exceeding max_chars when possible.

    If a single unit is too long, fall back to fixed-character splitting for that unit.
    """
    if max_chars <= 0:
        raise ValueError("max_chars must be positive")

    chunks = []
    current = ""

    for unit in units:
        if len(unit) > max_chars:
            if current:
                chunks.append(current.strip())
                current = ""
            chunks.extend(fixed_character_chunks(unit, chunk_size=max_chars, overlap=0))
            continue

        candidate = unit if not current else current + separator + unit
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            current = unit

    if current.strip():
        chunks.append(current.strip())

    return chunks


def sentence_chunks(text: str, max_chars: int = 180) -> list[str]:
    return pack_units(sentence_units(text), max_chars=max_chars, separator=" ")

sentence_chunk_records = build_chunk_records(
    cleaned_pages,
    lambda text: sentence_chunks(text, max_chars=180),
    "sentence_aware",
)

print("Sentence-aware chunks:", len(sentence_chunk_records))
show_chunks(sentence_chunk_records, limit=3)

Sentence-aware chunks: 12

data_pipeline_runbook::p1::c001 | 144 chars | 16 words
DATA PIPELINE RUNBOOK Pipeline: CUSTOMER_DAILY_LOAD Schedule: Daily at 02:00 UTC. Upstream dependency: CUSTOMER_RAW_INGEST. Normal operation: 1.

data_pipeline_runbook::p1::c002 | 171 chars | 22 words
Confirm the upstream ingestion completed successfully. 2. Confirm the expected business-date partition exists. 3. Start the downstream load after dependencies are healthy.

data_pipeline_runbook::p1::c003 | 23 chars | 3 words
Owner: Data Operations.


## 7. Strategy D — Paragraph-aware chunks

Paragraphs often keep related ideas together. This strategy splits on blank lines and packs whole paragraphs until the target length is reached. If one paragraph is too large, we fall back to sentence-aware splitting.

**Important PDF limitation:** a PDF extractor may return line breaks without reliable blank lines. If the original visual paragraph structure is lost during extraction, paragraph chunking cannot magically recover it. Better layout-aware parsing may be needed for complex PDFs.

In [6]:
def paragraph_chunks(text: str, max_chars: int = 180) -> list[str]:
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", text.strip())
        if paragraph and paragraph.strip()
    ]

    # If no blank-line separation survived PDF extraction, use sentence-aware units.
    if len(paragraphs) <= 1:
        return sentence_chunks(text, max_chars=max_chars)

    chunks = []
    current = ""

    for paragraph in paragraphs:
        if len(paragraph) > max_chars:
            if current:
                chunks.append(current.strip())
                current = ""
            chunks.extend(sentence_chunks(paragraph, max_chars=max_chars))
            continue

        candidate = paragraph if not current else current + "\n\n" + paragraph
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            current = paragraph

    if current.strip():
        chunks.append(current.strip())

    return chunks

paragraph_chunk_records = build_chunk_records(
    cleaned_pages,
    lambda text: paragraph_chunks(text, max_chars=180),
    "paragraph_aware",
)

print("Paragraph-aware chunks:", len(paragraph_chunk_records))
show_chunks(paragraph_chunk_records, limit=3)

Paragraph-aware chunks: 12

data_pipeline_runbook::p1::c001 | 144 chars | 16 words
DATA PIPELINE RUNBOOK Pipeline: CUSTOMER_DAILY_LOAD Schedule: Daily at 02:00 UTC. Upstream dependency: CUSTOMER_RAW_INGEST. Normal operation: 1.

data_pipeline_runbook::p1::c002 | 171 chars | 22 words
Confirm the upstream ingestion completed successfully. 2. Confirm the expected business-date partition exists. 3. Start the downstream load after dependencies are healthy.

data_pipeline_runbook::p1::c003 | 23 chars | 3 words
Owner: Data Operations.


## 8. Strategy E — Recursive character splitting

Recursive splitting is a useful general-purpose approach. Rather than cut at a fixed character index immediately, it tries preferred boundaries in order. Our teaching implementation tries:

1. blank lines (paragraph-like boundaries);
2. line breaks;
3. sentence punctuation followed by a space;
4. spaces (word boundaries);
5. individual characters as a last resort.

It then merges small pieces into chunks and carries a configurable overlap. This is the basic idea behind commonly used recursive splitters; LangChain's current documentation describes a similar strategy that tries paragraph breaks, line breaks, spaces, then individual characters. <https://github.com/langchain-ai/docs/blob/main/src/oss/integrations/splitters/recursive_text_splitter.mdx>

This is still a heuristic. It does not understand meaning; it simply prefers more natural boundaries.

In [7]:
def _split_keep_separator(text: str, separator: str) -> list[str]:
    """Split on separator while keeping each separator at the end of its piece."""
    if separator == "":
        return list(text)

    parts = text.split(separator)
    pieces = []
    for index, part in enumerate(parts):
        piece = part + (separator if index < len(parts) - 1 else "")
        if piece:
            pieces.append(piece)
    return pieces


def _recursive_units(text: str, chunk_size: int, separators: list[str]) -> list[str]:
    """Break oversized text into smaller units, preferring separators earlier in the list."""
    if len(text) <= chunk_size:
        return [text]

    chosen_index = None
    for index, separator in enumerate(separators):
        if separator == "" or separator in text:
            chosen_index = index
            break

    if chosen_index is None:
        return fixed_character_chunks(text, chunk_size=chunk_size, overlap=0)

    separator = separators[chosen_index]
    pieces = _split_keep_separator(text, separator)
    remaining_separators = separators[chosen_index + 1:]

    units = []
    for piece in pieces:
        if len(piece) > chunk_size:
            units.extend(_recursive_units(piece, chunk_size, remaining_separators))
        else:
            units.append(piece)

    return units


def recursive_character_chunks(
    text: str,
    chunk_size: int = 180,
    overlap: int = 30,
) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size")

    separators = ["\n\n", "\n", ". ", " ", ""]
    units = _recursive_units(text, chunk_size, separators)

    chunks = []
    current_units = []
    current_length = 0

    for unit in units:
        if not unit:
            continue

        if current_units and current_length + len(unit) > chunk_size:
            finished = "".join(current_units).strip()
            if finished:
                chunks.append(finished)

            # Keep a suffix of whole units as overlap; then ensure the next unit fits.
            while current_units and current_length > overlap:
                current_length -= len(current_units.pop(0))
            while current_units and current_length + len(unit) > chunk_size:
                current_length -= len(current_units.pop(0))

        current_units.append(unit)
        current_length += len(unit)

    tail = "".join(current_units).strip()
    if tail and (not chunks or tail != chunks[-1]):
        chunks.append(tail)

    return [chunk for chunk in chunks if chunk]

recursive_chunk_records = build_chunk_records(
    cleaned_pages,
    lambda text: recursive_character_chunks(text, chunk_size=180, overlap=30),
    "recursive_character",
)

print("Recursive chunks:", len(recursive_chunk_records))
show_chunks(recursive_chunk_records, limit=3)

Recursive chunks: 12

data_pipeline_runbook::p1::c001 | 141 chars | 15 words
DATA PIPELINE RUNBOOK
Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.
Normal operation:

data_pipeline_runbook::p1::c002 | 131 chars | 16 words
Normal operation:
1. Confirm the upstream ingestion completed successfully.
2. Confirm the expected business-date partition exists.

data_pipeline_runbook::p1::c003 | 84 chars | 12 words
3. Start the downstream load after dependencies are healthy.
Owner: Data Operations.


### What to inspect in recursive output

- Does it keep headings and list items together more often than fixed-character chunking?
- Does it still split some content awkwardly? It can—this is not semantic understanding.
- Do chunks overlap? Compare the end of one chunk with the start of the next chunk from the same page.
- Does every chunk still point to the correct source and page? Our wrapper preserves that metadata.

## 9. Strategy F — Token-based chunking (optional dependency)

Characters, words, and model tokens are different measures. A model consumes tokens, so token counts matter when fitting retrieved context into an embedding model or generation-model limit. There is no universal tokenization: `tiktoken` is designed for OpenAI models, so this experiment uses an OpenAI tokenizer as an example, not as a commitment to the final DocuTrace provider. See the <https://github.com/openai/tiktoken> documentation.

Run the following cell once. It installs the package into the **current notebook kernel's** environment.

In [8]:
%pip install tiktoken

  Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl (940 kB)

   ---------------------------------------- 0/2 [regex]
   ---------------------------------------- 0/2 [regex]
   ---------------------------------------- 0/2 [regex]
   -------------------- ------------------- 1/2 [tiktoken]
   -------------------- ------------------- 1/2 [tiktoken]
   ---------------------------------------- 2/2 [tiktoken]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


The implementation below uses token offsets to slice the original text at character boundaries, instead of blindly decoding arbitrary token windows. This helps avoid cutting a multi-byte Unicode character in half. A character boundary can occasionally require a small adjustment to the requested token window; we will inspect actual token counts rather than assuming every chunk is exact.

In [9]:
import tiktoken

# Illustrative tokenizer. We will select the correct tokenizer for the eventual model later.
encoding = tiktoken.get_encoding("cl100k_base")


def token_based_chunks(text: str, max_tokens: int = 35, overlap_tokens: int = 5) -> list[str]:
    if max_tokens <= 0:
        raise ValueError("max_tokens must be positive")
    if overlap_tokens < 0 or overlap_tokens >= max_tokens:
        raise ValueError("overlap_tokens must satisfy 0 <= overlap_tokens < max_tokens")

    token_ids = encoding.encode(text, disallowed_special=())
    if not token_ids:
        return []

    decoded_text, offsets = encoding.decode_with_offsets(token_ids)
    if decoded_text != text:
        raise ValueError("Tokenizer round-trip changed the input text; inspect encoding and input.")

    chunks = []
    start_token = 0
    total_tokens = len(token_ids)

    while start_token < total_tokens:
        # Move the start to the beginning of a character if tokens share one character offset.
        while start_token > 0 and offsets[start_token] == offsets[start_token - 1]:
            start_token -= 1

        end_token = min(start_token + max_tokens, total_tokens)

        # Avoid ending between tokens that map to the same Unicode character.
        if end_token < total_tokens:
            while end_token > start_token and offsets[end_token] == offsets[end_token - 1]:
                end_token -= 1
            # A single character may span multiple tokens; in that rare case include it intact.
            if end_token == start_token:
                end_token = min(start_token + max_tokens, total_tokens)
                while end_token < total_tokens and offsets[end_token] == offsets[end_token - 1]:
                    end_token += 1

        char_start = offsets[start_token]
        char_end = offsets[end_token] if end_token < total_tokens else len(decoded_text)
        chunk = decoded_text[char_start:char_end].strip()
        if chunk and (not chunks or chunk != chunks[-1]):
            chunks.append(chunk)

        if end_token >= total_tokens:
            break

        next_start = max(start_token + 1, end_token - overlap_tokens)
        while next_start > 0 and offsets[next_start] == offsets[next_start - 1]:
            next_start -= 1
        if next_start <= start_token:
            next_start = end_token  # safeguard against a non-advancing loop
        start_token = next_start

    return chunks

# First, inspect token counts on a sample page.
sample_text = cleaned_pages[0]["text"]
print("Character count:", len(sample_text))
print("Whitespace-delimited word count:", len(sample_text.split()))
print("Token count under cl100k_base:", len(encoding.encode(sample_text, disallowed_special=())))

sample_token_chunks = token_based_chunks(sample_text, max_tokens=35, overlap_tokens=5)
for i, chunk in enumerate(sample_token_chunks, start=1):
    print(f"\nToken chunk {i} | {len(encoding.encode(chunk, disallowed_special=()))} tokens | {len(chunk)} chars")
    print(chunk)

Character count: 340
Whitespace-delimited word count: 41
Token count under cl100k_base: 72

Token chunk 1 | 35 tokens | 130 chars
DATA PIPELINE RUNBOOK
Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.
Normal

Token chunk 2 | 35 tokens | 196 chars
_RAW_INGEST.
Normal operation:
1. Confirm the upstream ingestion completed successfully.
2. Confirm the expected business-date partition exists.
3. Start the downstream load after dependencies are

Token chunk 3 | 13 tokens | 71 chars
downstream load after dependencies are healthy.
Owner: Data Operations.


**Key lesson:** do not treat `cl100k_base` as universal. When we choose the embedding and generation models for DocuTrace, we will check their tokenization and limits. Also, splitting by token count alone does not preserve sentence or paragraph boundaries; token-based and structure-aware splitting can be combined.

## 10. Compare all strategies

Use the same source pages and compare the number and average size of the resulting chunks. Counts are descriptive, not a quality score: more chunks does not automatically mean better retrieval.

In [10]:
def summarize_chunks(name: str, chunks: list[dict]) -> dict:
    lengths = [chunk["char_count"] for chunk in chunks]
    words = [chunk["word_count"] for chunk in chunks]
    return {
        "strategy": name,
        "chunks": len(chunks),
        "avg_chars": round(statistics.mean(lengths), 1) if lengths else 0,
        "max_chars": max(lengths, default=0),
        "avg_words": round(statistics.mean(words), 1) if words else 0,
    }

strategy_results = {
    "fixed_character": char_chunks,
    "fixed_word": word_chunks,
    "sentence_aware": sentence_chunk_records,
    "paragraph_aware": paragraph_chunk_records,
    "recursive_character": recursive_chunk_records,
}

comparison = [summarize_chunks(name, records) for name, records in strategy_results.items()]
for row in comparison:
    print(row)

{'strategy': 'fixed_character', 'chunks': 12, 'avg_chars': 146.7, 'max_chars': 180, 'avg_words': 21.5}
{'strategy': 'fixed_word', 'chunks': 8, 'avg_chars': 206.2, 'max_chars': 290, 'avg_words': 29.4}
{'strategy': 'sentence_aware', 'chunks': 12, 'avg_chars': 126.2, 'max_chars': 180, 'avg_words': 17.9}
{'strategy': 'paragraph_aware', 'chunks': 12, 'avg_chars': 126.2, 'max_chars': 180, 'avg_words': 17.9}
{'strategy': 'recursive_character', 'chunks': 12, 'avg_chars': 127.8, 'max_chars': 179, 'avg_words': 18.1}


## 11. Experiment — change chunk size

Do not immediately decide that one chunk size is “best”. Run the same recursive strategy with several small sizes, then observe how the number of chunks and average size changes. In a later notebook, we will evaluate whether the change helps retrieval quality.

In [11]:
for size in [100, 180, 260]:
    experiment_chunks = build_chunk_records(
        cleaned_pages,
        lambda text, size=size: recursive_character_chunks(
            text, chunk_size=size, overlap=min(20, size - 1)
        ),
        f"recursive_{size}",
    )
    print(summarize_chunks(f"chunk_size={size}", experiment_chunks))

{'strategy': 'chunk_size=100', 'chunks': 23, 'avg_chars': 67.9, 'max_chars': 99, 'avg_words': 9.7}
{'strategy': 'chunk_size=180', 'chunks': 12, 'avg_chars': 127.8, 'max_chars': 179, 'avg_words': 18.1}
{'strategy': 'chunk_size=260', 'chunks': 8, 'avg_chars': 189.9, 'max_chars': 255, 'avg_words': 26.9}


## 12. Experiment — change overlap

Overlap can preserve context across boundaries, but it also duplicates text. Compare the number of chunks and total stored characters for the same chunk size with different overlaps.

In [12]:
for overlap in [0, 15, 35]:
    experiment_chunks = build_chunk_records(
        cleaned_pages,
        lambda text, overlap=overlap: recursive_character_chunks(
            text, chunk_size=180, overlap=overlap
        ),
        f"recursive_overlap_{overlap}",
    )
    total_chars = sum(chunk["char_count"] for chunk in experiment_chunks)
    print({
        "overlap": overlap,
        "chunks": len(experiment_chunks),
        "total_characters_across_chunks": total_chars,
    })

{'overlap': 0, 'chunks': 12, 'total_characters_across_chunks': 1515}
{'overlap': 15, 'chunks': 12, 'total_characters_across_chunks': 1515}
{'overlap': 35, 'chunks': 12, 'total_characters_across_chunks': 1533}


## 13. Metadata and content checks

Before a chunk can be embedded, its provenance should remain intact. Verify that every chunk refers to an input source/page pair and contains non-empty text.

In [13]:
source_page_pairs = {
    (page["source"], page["page_number"])
    for page in cleaned_pages
}

for strategy_name, records in strategy_results.items():
    assert all(record["text"].strip() for record in records), strategy_name
    assert all(
        (record["source"], record["page_number"]) in source_page_pairs
        for record in records
    ), strategy_name
    assert len({record["chunk_id"] for record in records}) == len(records), strategy_name

print("Passed: chunks are non-empty, IDs are unique within each strategy, and source/page metadata is valid.")

Passed: chunks are non-empty, IDs are unique within each strategy, and source/page metadata is valid.


## 14. Save one experiment output

We are saving the recursive strategy as a **temporary experiment artifact**, not yet declaring it the winner. Later, evaluation will help us decide which chunking configuration performs best for our documents and questions.

In [14]:
output_path = PROJECT_ROOT / "data" / "processed" / "chunks_recursive_experiment.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as file:
    json.dump(recursive_chunk_records, file, ensure_ascii=False, indent=2)

print("Saved:", output_path)
print("Saved chunk records:", len(recursive_chunk_records))

Saved: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\data\processed\chunks_recursive_experiment.json
Saved chunk records: 12


## 15. Quick review — answer in your own words

1. Why might fixed-character chunking cut information at an awkward boundary?
2. What is the purpose of overlap? What is its cost?
3. How is a token different from a character or a whitespace-delimited word?
4. Why does every chunk preserve the original PDF filename and page number?
5. Does the strategy producing the most chunks necessarily provide the best answers? Why not?
6. Why might paragraph-aware splitting struggle with some PDFs?

### What comes next?

Next we will move from chunk preparation to **embeddings**. Before doing so, we should inspect and compare the chunks you created, then discuss structure-aware chunking and semantic chunking. Semantic chunking depends on measuring meaning using embeddings, so it makes sense to implement it after the embedding concepts are clear.

**Do not choose a final chunk size based only on this synthetic dataset.** We will build an evaluation set with questions and relevant evidence before making that decision.